In [2]:
import torch
from utils import load_splits
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

/home/default/projects/fake_news_classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [4]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

1
2
3


In [6]:

def init_model(model_name):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "True", 1:"Fake"},
        label2id={"True": 0, "Fake": 0}
    )
    return model


In [7]:
from datasets import Dataset
from transformers import AutoTokenizer

def tokenize_dataset(x, y, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    dataset = Dataset.from_dict({
        "text": x.fillna("").astype(str).tolist(),
        "labels": y.astype(int).tolist()
    })

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=256
        )

    return dataset.map(
        tokenize,
        batched=True
    )


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [21]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

import torch
import gc
import json
from pathlib import Path


def training_loop(ds_train, ds_val, learning_rate, model_name, per_device_train_batch_size, gradient_accumulation_step):
    print(f"Training with learning rate: {learning_rate}")

    model = init_model(model_name)

    training_args = TrainingArguments(
        output_dir=f"models/{model_name}_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=gradient_accumulation_step,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=5,
        logging_strategy="epoch",
        report_to="none",
        seed=42,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    del trainer, model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

In [ ]:

all_results = []
model = "distilbert-base-uncased"

for learning_rate in [1e-7]:
    ds_train = tokenize_dataset(X_train, y_train, model)
    ds_val =  tokenize_dataset(X_val, y_val, model)     
    all_results.extend(training_loop(ds_train, ds_val, learning_rate, model, 4, 4))



In [29]:
# test
import json
from pathlib import Path
for lr in [1e-5, 2e-5 , 3e-5]:
    model_name = f"bert-base-uncased_{lr}"
    path = Path(f'models/{model_name}/checkpoint-13615/trainer_state.json')

    with open(path, 'r') as f:
        trainer_state = json.load(f)


    log_history = trainer_state["log_history"]

    results = []
    for log in log_history:
        if "loss" in log:
            epoch_results = log
        if "eval_f1_macro" in log:
            epoch_results.update(log)
            epoch_results["learning_rate"] = lr
            results.append(epoch_results)
        
    print(results)
    log_path = f"logs/{model_name}.json"

    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4)

[{'epoch': 1.0, 'grad_norm': 0.45982691645622253, 'learning_rate': 1e-05, 'loss': 1.8025546989763128, 'step': 2723, 'eval_accuracy': 0.9368906032358298, 'eval_f1_macro': 0.9363577362317469, 'eval_loss': 0.18825526535511017, 'eval_precision_macro': 0.9351364491644607, 'eval_recall_macro': 0.9383330507810208, 'eval_runtime': 54.4626, 'eval_samples_per_second': 171.365, 'eval_steps_per_second': 42.855}, {'epoch': 2.0, 'grad_norm': 34.70402145385742, 'learning_rate': 1e-05, 'loss': 1.3708847555832262, 'step': 5446, 'eval_accuracy': 0.9418193506910961, 'eval_f1_macro': 0.9408112799883533, 'eval_loss': 0.2971939444541931, 'eval_precision_macro': 0.9434225434416619, 'eval_recall_macro': 0.9389273173466692, 'eval_runtime': 51.5175, 'eval_samples_per_second': 181.162, 'eval_steps_per_second': 45.305}, {'epoch': 3.0, 'grad_norm': 0.005152707453817129, 'learning_rate': 1e-05, 'loss': 1.0025023741622292, 'step': 8169, 'eval_accuracy': 0.94492660452159, 'eval_f1_macro': 0.944246228420172, 'eval_los